# Hashformers MCP + Agent Skill with Codex and Claude Code

This tutorial runs the Hashformers MCP server inside Google Colab and connects it to both [Codex CLI](https://learn.chatgpt.com/docs/developer-commands) and [Claude Code](https://code.claude.com/docs/en/overview). You will:

1. clone Hashformers from GitHub and install `hashformers[mcp]` from that checkout;
2. install and authenticate Codex CLI and Claude Code;
3. install the repository's `segment-hashtags` Agent Skill;
4. register the local stdio MCP server with both clients; and
5. exercise the interactive, resumable-file, and deferred-model workflows described in the README.

> **Runtime:** choose **Runtime → Change runtime type → T4 GPU** before running the notebook. Colab VMs are temporary; repeat the authentication cells after a runtime reset. Never paste API keys into a notebook that will be shared.


## 1. Check the Colab runtime

The agents themselves are command-line clients. Hashformers uses the Colab GPU when the MCP server loads the Transformer model.


In [1]:
import platform
import subprocess

print("Python:", platform.python_version())
subprocess.run(["node", "--version"], check=True)
subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    check=True,
)


Python: 3.12.13


CompletedProcess(args=['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], returncode=0)

## 2. Clone and install from Git

For development validation, install Hashformers directly from the checked-out Git branch rather than PyPI. The MCP extra supplies the MCP SDK and the `hashformers-mcp` entry point. The saved output was captured from the feature branch used to validate this tutorial; the released cell defaults to `master`.


In [2]:
REPOSITORY = "https://github.com/ruanchaves/hashformers.git"
BRANCH = "master"  # Use your working branch when validating a PR.

!git clone --branch {BRANCH} --single-branch {REPOSITORY} /content/hashformers
%cd /content/hashformers
!git rev-parse --abbrev-ref HEAD
!git rev-parse HEAD
!git status --short --branch


Cloning into '/content/hashformers'...
remote: Enumerating objects: 3536, done.
remote: Counting objects: 100% (191/191), done.
remote: Compressing objects: 100% (135/135), done.
remote: Total 3536 (delta 63), reused 95 (delta 52), pack-reused 3345 (from 2)
Receiving objects: 100% (3536/3536), 24.67 MiB | 17.96 MiB/s, done.
Resolving deltas: 100% (2188/2188), done.
/content/hashformers
agent/colab-agent-mcp-tutorial
4ac60e9269dd7e8dc1ea1a84c45b375319df4c3e
## agent/colab-agent-mcp-tutorial...origin/agent/colab-agent-mcp-tutorial


In [3]:
!python -m pip install -q -e ".[mcp]"
!python -m pip show hashformers | grep -E '^(Name|Version|Editable project location):'
!hashformers-mcp --help | sed -n '1,28p'


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 350.0/350.0 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.6/69.6 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.7/51.7 kB 5.2 MB/s eta 0:00:00
Name: hashformers
Version: 3.0.0
Editable project location: /content/hashformers/src
usage: hashformers-mcp [-h] [--defer-model-selection]
                       [--model SEGMENTER_MODEL]
                       [--segmenter-model-type SEGMENTER_MODEL_TYPE]
                       [--device SEGMENTER_DEVICE]
                       [--batch-size SEGMENTER_BATCH_SIZE]
                       [--max-batch-size SEGMENTER_MAX_BATCH_SIZE]
                       [--reranker-model RERANKER_MODEL]
                       [--reranker-model-type RERANKER_MODEL_TYPE]
                       [--reranker-device RERANKER_DEVICE]
                       [--r

## 3. Install Codex CLI and Claude Code

Both clients are installed in the Colab VM. The commands below use their supported npm packages, which is convenient in Colab's preconfigured Node.js environment. See the current [Codex authentication guide](https://learn.chatgpt.com/docs/auth) and [Claude Code setup guide](https://code.claude.com/docs/en/installation) if a later CLI release changes the flow.


In [4]:
!npm install -g @openai/codex @anthropic-ai/claude-code
!codex --version
!claude --version


⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦
added 4 packages in 10s
⠦npm notice
npm notice New major version of npm available! 10.8.2 -> 12.0.2
npm notice Changelog: https://github.com/npm/cli/releases/tag/v12.0.2
npm notice To update run: npm install -g npm@12.0.2
npm notice
⠦codex-cli 0.146.0
2.1.197 (Claude Code)


## 4. Authenticate the two clients

Run these cells **one at a time**. For Codex, open the printed device-login URL and enter the one-time code. For Claude Code, open the printed URL; if the browser displays a login code instead of redirecting to Colab, paste that code into the cell prompt.

> The published copy intentionally clears the transient login-cell outputs. The following status check records only whether each login succeeded.


In [ ]:
!codex login --device-auth


In [ ]:
import pexpect

claude_login = pexpect.spawn(
    "claude", ["auth", "login"], encoding="utf-8", timeout=300
)
claude_login.setecho(False)
claude_login.expect(r"If the browser didn't open, visit: (https://\S+)")
authorization_url = claude_login.match.group(1)
print("Open this URL in your browser:\n", authorization_url)
claude_login.expect("Paste code here if prompted >")
authorization_code = input("Paste the code from the browser here: ")
claude_login.sendline(authorization_code)
claude_login.expect(pexpect.EOF)
claude_login.close()
if claude_login.exitstatus != 0:
    raise RuntimeError("Claude Code authentication failed")
print("Claude Code login completed")
del authorization_code

In [10]:
import subprocess

auth_checks = {
    "Codex": ["codex", "login", "status"],
    "Claude Code": ["claude", "auth", "status"],
}
for client, command in auth_checks.items():
    completed = subprocess.run(
        command, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
    )
    print(f"{client} authenticated: {completed.returncode == 0}")

Codex authenticated: True
Claude Code authenticated: True


## 5. Install the Agent Skill

The MCP server provides bounded tools and enforces their contracts. The `segment-hashtags` skill teaches each agent when and how to combine those tools. Codex discovers personal skills under `~/.agents/skills`; Claude Code uses `~/.claude/skills`.


In [11]:
!mkdir -p ~/.agents/skills ~/.claude/skills
!cp -R .agents/skills/segment-hashtags ~/.agents/skills/
!cp -R .agents/skills/segment-hashtags ~/.claude/skills/
!test -f ~/.agents/skills/segment-hashtags/SKILL.md && echo "Codex skill installed"
!test -f ~/.claude/skills/segment-hashtags/SKILL.md && echo "Claude Code skill installed"


Codex skill installed
Claude Code skill installed


## 6. Register the Hashformers MCP server

These are the README's stdio registration commands. This tutorial also authorizes one explicit Colab directory for the resumable file example and enables automatic CUDA microbatch tuning. The clients start `hashformers-mcp` when an agent session begins.


In [12]:
from pathlib import Path

DATA_DIRECTORY = Path("/content/hashformers/tutorial_data")
DATA_DIRECTORY.mkdir(exist_ok=True)

!codex mcp add hashformers -- hashformers-mcp --model distilgpt2 --batch-size auto --file-root {DATA_DIRECTORY}
!claude mcp add --transport stdio --scope user hashformers -- hashformers-mcp --model distilgpt2 --batch-size auto --file-root {DATA_DIRECTORY}


Added global MCP server 'hashformers'.
Added stdio MCP server hashformers with command: hashformers-mcp --model distilgpt2 --batch-size auto --file-root /content/hashformers/tutorial_data to user config
File modified: /root/.claude.json


In [13]:
!codex mcp list
!claude mcp get hashformers


Name         Command          Args                                                                                 Env  Cwd  Status   Auth       
hashformers  hashformers-mcp  --model distilgpt2 --batch-size auto --file-root /content/hashformers/tutorial_data  -    -    enabled  Unsupported
78hashformers:
Scope:Userconfig(availableinallyourprojects)
Status:✔Connected
Type:stdio
Command:hashformers-mcp
Args:--modeldistilgpt2--batch-sizeauto--file-root
/content/hashformers/tutorial_data
Environment:

Toremovethisserver,run:claudemcpremovehashformers-suser
(B[>4m[<u78(B[>4m[<u78]0;

## 7. Segment hashtags interactively

First ask Codex, then Claude Code, to use the installed skill and the MCP tool. The first model-backed call downloads and caches `distilgpt2`; later calls reuse the Hugging Face cache.


In [15]:
CODEX_PROMPT = (
    "Use the segment-hashtags skill and the Hashformers MCP server to segment "
    "#weneedanationalpark and #icecold. Return up to three candidates for each "
    "hashtag. Do not estimate the segmentations yourself."
)
!codex --ask-for-approval never exec --ephemeral --sandbox read-only "{CODEX_PROMPT}"

OpenAI Codex v0.146.0
--------
workdir: /content/hashformers
model: gpt-5.6-sol
provider: openai
approval: never
sandbox: read-only
reasoning effort: none
reasoning summaries: none
session id: 019fcc77-8e53-7c10-b8fa-aa7a53cc2ebb
--------
user
Use the segment-hashtags skill and the Hashformers MCP server to segment #weneedanationalpark and #icecold. Return up to three candidates for each hashtag. Do not estimate the segmentations yourself.
codex
I’m using the segment-hashtags skill because it provides the MCP-only workflow for generating and ranking candidates without guessing locally.
exec
/bin/bash -lc "sed -n '1,240p' /content/hashformers/.agents/skills/segment-hashtags/SKILL.md" in /content/hashformers
 succeeded in 0ms:
---
name: segment-hashtags
description: Segment hashtags, sample and process hashtag files without copying their contents into agent context, discover and configure language-appropriate public Hub models, and rank precomputed segmentation candidates with the Hashfo

In [17]:
CLAUDE_PROMPT = (
    "Use the segment-hashtags skill and the Hashformers MCP server to segment "
    "#weneedanationalpark and #icecold. Return up to three candidates for each "
    "hashtag. Do not estimate the segmentations yourself."
)
!claude -p --allowedTools "mcp__hashformers__segment_hashtags" --max-turns 8 "{CLAUDE_PROMPT}"

Segmented via the Hashformers MCP server (segmenter model: `distilgpt2`, lower score = better):

**#weneedanationalpark**
1. **we need a national park** (score 35.09)
2. we needa national park (score 41.57)
3. we need a nationalpark (score 42.67)

**#icecold**
1. **ice cold** (score 27.65)
2. icecold (score 28.08)
3. icec old (score 36.74)

Top-ranked candidate is bolded for each.
(B[>4m[<u78]0;

## 8. Process a CSV with the resumable file workflow

The agent passes paths—not the whole dataset—to `start_hashtag_file_job`, then calls `continue_hashtag_file_job` until the checkpoint reports completion. The MCP server preserves source order and duplicates in the JSON Lines output.


In [18]:
CSV_PATH = DATA_DIRECTORY / "hashtags.csv"
OUTPUT_PATH = DATA_DIRECTORY / "segmented.jsonl"
CSV_PATH.write_text(
    "hashtag\n#blacklivesmatter\n#myoldphonesucks\n#icecold\n#icecold\n",
    encoding="utf-8",
)
print(CSV_PATH.read_text(encoding="utf-8"))


hashtag
#blacklivesmatter
#myoldphonesucks
#icecold
#icecold



In [20]:
FILE_PROMPT = (
    f"Use the segment-hashtags skill and Hashformers to segment the hashtags in {CSV_PATH}. "
    f"Save the results to {OUTPUT_PATH} and continue until the job is complete. "
    "Do not open or copy the complete input file into your context."
)
!codex -c 'mcp_servers.hashformers.tools.start_hashtag_file_job.approval_mode="approve"' -c 'mcp_servers.hashformers.tools.continue_hashtag_file_job.approval_mode="approve"' --ask-for-approval never exec --ephemeral --sandbox workspace-write "{FILE_PROMPT}"

OpenAI Codex v0.146.0
--------
workdir: /content/hashformers
model: gpt-5.6-sol
provider: openai
approval: never
sandbox: workspace-write [workdir, /tmp, $TMPDIR]
reasoning effort: none
reasoning summaries: none
session id: 019fcc7b-2a7e-7762-bd3f-cca3d5fcbf5c
--------
user
Use the segment-hashtags skill and Hashformers to segment the hashtags in /content/hashformers/tutorial_data/hashtags.csv. Save the results to /content/hashformers/tutorial_data/segmented.jsonl and continue until the job is complete. Do not open or copy the complete input file into your context.
codex
I’m using the `segment-hashtags` skill because it provides the streaming workflow for bulk segmentation without loading the full CSV into context. I’ll read its instructions, then run the job through completion and verify the output.
exec
/bin/bash -lc "sed -n '1,240p' .agents/skills/segment-hashtags/SKILL.md" in /content/hashformers
 succeeded in 0ms:
---
name: segment-hashtags
description: Segment hashtags, sample an

In [22]:
import json

records = [json.loads(line) for line in OUTPUT_PATH.read_text().splitlines()]
print(f"Output records: {len(records)}")
for record in records:
    print(record["input"], "->", record["selected_segmentation"])

Output records: 4
#blacklivesmatter -> blacklivesmatter
#myoldphonesucks -> my old phone sucks
#icecold -> ice cold
#icecold -> ice cold


## 9. Configure deferred model selection for an unknown language

A server started with `--defer-model-selection` loads no Transformer at startup. The skill can sample at most 20 distinct local examples, infer a language, ask Hashformers for a bounded public Hugging Face shortlist, and configure one exact model revision before inference. A running server cannot hot-swap models, so this example replaces Codex's earlier registration and starts a fresh agent session.


In [23]:
UNKNOWN_LANGUAGE_PATH = DATA_DIRECTORY / "unknown_language.csv"
UNKNOWN_OUTPUT_PATH = DATA_DIRECTORY / "unknown_language_segmented.jsonl"
UNKNOWN_LANGUAGE_PATH.write_text(
    "hashtag\n#saopaulomeuamor\n#futebolbrasileiro\n#cafecomleite\n",
    encoding="utf-8",
)

!codex mcp remove hashformers
!codex mcp add hashformers -- hashformers-mcp --defer-model-selection --file-root {DATA_DIRECTORY}
!codex mcp get hashformers


Removed global MCP server 'hashformers'.
Added global MCP server 'hashformers'.
hashformers
  enabled: true
  transport: stdio
  command: hashformers-mcp
  args: --defer-model-selection --file-root /content/hashformers/tutorial_data
  cwd: -
  env: -
  remove: codex mcp remove hashformers


In [24]:
DEFERRED_PROMPT = (
    f"Use the segment-hashtags skill and Hashformers to sample {UNKNOWN_LANGUAGE_PATH}, "
    "identify its language, select a compatible public Hugging Face segmenter at an exact "
    f"revision, and segment the file to {UNKNOWN_OUTPUT_PATH}. Continue until the job is complete."
)
!codex -c 'mcp_servers.hashformers.tools.configure_models.approval_mode="approve"' -c 'mcp_servers.hashformers.tools.start_hashtag_file_job.approval_mode="approve"' -c 'mcp_servers.hashformers.tools.continue_hashtag_file_job.approval_mode="approve"' --ask-for-approval never exec --ephemeral --sandbox workspace-write "{DEFERRED_PROMPT}"

OpenAI Codex v0.146.0
--------
workdir: /content/hashformers
model: gpt-5.6-sol
provider: openai
approval: never
sandbox: workspace-write [workdir, /tmp, $TMPDIR]
reasoning effort: none
reasoning summaries: none
session id: 019fcc7d-742a-7531-8f04-e9cbcf1bbe6a
--------
user
Use the segment-hashtags skill and Hashformers to sample /content/hashformers/tutorial_data/unknown_language.csv, identify its language, select a compatible public Hugging Face segmenter at an exact revision, and segment the file to /content/hashformers/tutorial_data/unknown_language_segmented.jsonl. Continue until the job is complete.
codex
I’m using the `segment-hashtags` skill because this is its full unknown-language workflow: sample safely, identify the language, pin a compatible public model revision, and process the file. I’ll follow it through to a verified JSONL output.
exec
/bin/bash -lc "sed -n '1,240p' .agents/skills/segment-hashtags/SKILL.md" in /content/hashformers
 succeeded in 0ms:
---
name: segment-

## What to take to a local machine

Outside Colab, install a released build with `pip install "hashformers[mcp]"`, copy the skill to the client-specific personal skill directory, and use the same `codex mcp add` or `claude mcp add` commands. Authorize only the file roots the agent actually needs. Run `hashformers-mcp --help` for model, reranker, device, batch-size, and file-access options.
